# Day 04: Convolutional Networks — Fashion-MNIST

Welcome to practical 4! Learn spatial feature extraction with a small CNN.

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Fill in methods marked `# YOUR CODE HERE`.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

A 2-D convolution applies a learned filter across spatial locations: $(f * k)[i,j] = \sum_{u,v} f[i+u,j+v]\, k[u,v]$. Stacking conv–ReLU–pool blocks builds translation-equivariant representations.

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.ToTensor()])
fashion_train = datasets.FashionMNIST("/tmp/fashion", train=True, download=True, transform=transform)
fashion_test = datasets.FashionMNIST("/tmp/fashion", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(fashion_train, range(3000)), batch_size=64, shuffle=True)
test_loader = DataLoader(Subset(fashion_test, range(1000)), batch_size=256)

### Exercise 4.1: Small CNN

**Your job**: Build `TinyCNN` with two conv blocks.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = TinyCNN().to(device)

### Exercise 4.2: Train for 2 epochs

**Your job**: Use Adam and cross-entropy.

In [ ]:
def run_training(model, loader, optimizer, epochs=2):
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # YOUR CODE HERE
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f"Epoch {epoch+1} avg loss: {running/len(loader):.4f}")

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
run_training(model, train_loader, opt)

### Exercise 4.3: Visualize filters

**Your job**: Plot the first-layer conv filters.

In [ ]:
w = model.features[0].weight.detach().cpu()
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    if i < w.size(0):
        ax.imshow(w[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("First conv layer filters")
plt.tight_layout()
plt.show()

### Exercise 4.4: Test accuracy

**Your job**: Evaluate on the held-out subset.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    ok, n = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        ok += (model(x).argmax(1) == y).sum().item()
        n += y.size(0)
    return ok / n

print("Test accuracy:", accuracy(model, test_loader))

## Reflection (Day 4)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (convolutional networks)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?